In [9]:
import os

import pandas as pd

import kagglehub

from kagglehub import KaggleDatasetAdapter



# Kaggle dataset + file inside the dataset (must include extension)

dataset_ref = "sirishk99/cmrti-placement-questions-dataset"

file_path = "undergrad-cs-questions-data.parquet"



df = kagglehub.load_dataset(

    KaggleDatasetAdapter.PANDAS,

    dataset_ref,

    file_path,

)



columns_to_drop = [

    col for col in ["relative_path", "source_type", "file_size", "file_size_kb"]

    if col in df.columns

]

df = df.drop(columns=columns_to_drop)



print("Loaded shape:", df.shape)

print(df.head())



data = df.to_dict(orient="records")

print("Sample record:", data[0])


/tmp/ipykernel_164465/1921347449.py:19: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Loaded shape: (50, 3)
                                            filename  \
0                             PTQB 2025-26_AInDS.csv   
1  CSE_ technical and programming questions place...   
2                             ISE PTQB 2025-2026.csv   
3                         ISE PTQB 2024-2025 (1).csv   
4                             PTQB 2024-25_AInDS.csv   

                                                text  text_length  
0              company name                      ...        76811  
1              company name                      ...       165179  
2       company name                             ...        19231  
3                        company name            ...       261225  
4             company name                       ...        37907  
Sample record: {'filename': 'PTQB 2025-26_AInDS.csv', 'text': '            company name                                                                                                                                                  

In [10]:
#Chunk the data
def chunk_data(text, chunk_size=200, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

chunked_data = []
for record in data:
    text = record['text']
    chunks = chunk_data(text)
    for i, chunk in enumerate(chunks):
        chunked_record = record.copy()
        chunked_record['text'] = chunk
        chunked_record['chunk_index'] = i
        chunked_data.append(chunked_record)
print("Total chunks created:", len(chunked_data))
print("Sample chunked record:", chunked_data[0])

Total chunks created: 10186
Sample chunked record: {'filename': 'PTQB 2025-26_AInDS.csv', 'text': '            company name                                                                                                                                                                                ', 'text_length': 76811, 'chunk_index': 0}


### Initialize chroma vector database client and SentenceTransformer embedding encoder

In [ ]:
# initialize ChromaDB in memory and create collection
import chromadb
from sentence_transformers import SentenceTransformer
import torch


client = chromadb.EphemeralClient()
collection = client.get_or_create_collection(
    name="placement_questions",
    metadata={"hnsw:space": "cosine"},
)

device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
print("Using device:", device)

# update collection with chunked data
texts = [record["text"] for record in chunked_data]
embeddings = embedding_model.encode(texts, batch_size=32, show_progress_bar=True).tolist()
added_ids = []

for idx, (record, embedding) in enumerate(zip(chunked_data, embeddings)):
    chunk_id = f"{record.get('filename', 'document')}_chunk_{record['chunk_index']}_{idx}"
    metadata = {k: v for k, v in record.items() if k != "text"}
    collection.add(
        documents=[record["text"]],
        metadatas=[metadata],
        ids=[chunk_id],
        embeddings=[embedding],
    )
    added_ids.append(chunk_id)

print("Total documents in collection:", collection.count())
if added_ids:
    print("Sample document from collection:", collection.get(ids=[added_ids[0]]))


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

KeyboardInterrupt: 